# 28_PF001_PF003 - Congelamiento de datasets, splits e inventario

Este notebook cubre los tickets **PF-001 a PF-003** del backlog de producto final:

- **PF-001:** estandarizar estructura de datos final.
- **PF-002:** congelar splits finales.
- **PF-003:** generar inventario de datasets, licencias y evidencia.

No entrena modelos. Su función es dejar una base reproducible para que los entrenamientos finales, endpoints y demo usen las mismas rutas, los mismos splits y la misma documentación de datos.

> Criterio metodológico: este notebook crea evidencia versionable y evita que los próximos pasos dependan de decisiones implícitas o rutas sueltas de Colab.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import os, json, hashlib, datetime, re, math
import pandas as pd
import numpy as np

PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
REPO_ROOT = PFI_ROOT / "repo"
DATA_ROOT = PFI_ROOT / "data"
RESULTS_ROOT = PFI_ROOT / "results"
DOCS_ROOT = PFI_ROOT / "docs"
MODELS_ROOT = PFI_ROOT / "models"

PF_ROOT = RESULTS_ROOT / "PF001_PF003_dataset_freeze"
PF_ROOT.mkdir(parents=True, exist_ok=True)

REPO_CONFIG_ROOT = REPO_ROOT / "config"
REPO_DOCS_ROOT = REPO_ROOT / "docs"
REPO_BACKLOG_ROOT = REPO_ROOT / "backlogProducto"
for p in [REPO_CONFIG_ROOT, REPO_DOCS_ROOT, REPO_BACKLOG_ROOT, DOCS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

SPIDER_ROOT = DATA_ROOT / "SPIDER"
AXIAL_ROOT = DATA_ROOT / "AXIAL_ALKAFRI"
SSMSPINE_ROOT = DATA_ROOT / "SSMSPINE"

print("PFI_ROOT:", PFI_ROOT, "| exists:", PFI_ROOT.exists())
print("REPO_ROOT:", REPO_ROOT, "| exists:", REPO_ROOT.exists())
print("SPIDER_ROOT:", SPIDER_ROOT, "| exists:", SPIDER_ROOT.exists())
print("AXIAL_ROOT:", AXIAL_ROOT, "| exists:", AXIAL_ROOT.exists())
print("PF_ROOT:", PF_ROOT)

PFI_ROOT: /content/drive/MyDrive/PFI_MVP | exists: True
REPO_ROOT: /content/drive/MyDrive/PFI_MVP/repo | exists: True
SPIDER_ROOT: /content/drive/MyDrive/PFI_MVP/data/SPIDER | exists: True
AXIAL_ROOT: /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI | exists: True
PF_ROOT: /content/drive/MyDrive/PFI_MVP/results/PF001_PF003_dataset_freeze


## PF-001 - Estructura canónica de datos

Esta sección define rutas oficiales para la etapa final. No mueve archivos pesados; genera una configuración versionable para que el resto del producto use las mismas ubicaciones.

In [ ]:
freeze_timestamp = datetime.datetime.now().isoformat(timespec="seconds")

canonical_config = {
    "freeze_name": "PF001_PF003_dataset_freeze",
    "freeze_timestamp": freeze_timestamp,
    "pfi_root": str(PFI_ROOT),
    "repo_root": str(REPO_ROOT),
    "data_root": str(DATA_ROOT),
    "results_root": str(RESULTS_ROOT),
    "docs_root": str(DOCS_ROOT),
    "models_root": str(MODELS_ROOT),
    "datasets": {
        "spider_sagittal": {
            "root": str(SPIDER_ROOT),
            "status": "primary_sagittal_dataset",
            "role": "modelo sagital multiclase sobre vértebras, canal y discos",
            "expected_use": "training/evaluation/inference_sagittal",
        },
        "alkafri_sudirman_axial": {
            "root": str(AXIAL_ROOT),
            "status": "complementary_axial_dataset",
            "role": "modelo axial T2 sobre regiones axiales anotadas",
            "expected_use": "training/evaluation/inference_axial",
        },
        "ssmspine_optional": {
            "root": str(SSMSPINE_ROOT),
            "status": "optional_reference_if_present",
            "role": "no usado como núcleo del MVP final salvo decisión posterior",
            "expected_use": "reference_or_future_work",
        },
    },
    "final_models": {
        "sagittal_spider": str(MODELS_ROOT / "E12_sagittal_multiclass_final_best.pt"),
        "axial_t2_alkafri": str(MODELS_ROOT / "E10_axial_t2_final_training_clean_best.pt"),
    },
    "final_results_expected": {
        "E13_multiplanar": str(RESULTS_ROOT / "E13_multiplanar_inference_pipeline"),
        "E14_agent": str(RESULTS_ROOT / "E14_ai_agent_orchestrator"),
        "E15_ai_service": str(RESULTS_ROOT / "E15_backend_mvp_translation"),
        "E16_backend_smoke": str(RESULTS_ROOT / "E16_backend_integration_smoke"),
        "E17_spring_bridge": str(RESULTS_ROOT / "E17_spring_boot_bridge_scaffold"),
        "E18_frontend": str(RESULTS_ROOT / "E18_frontend_worklist_ui"),
        "E19_consolidation": str(RESULTS_ROOT / "E19_mvp_technical_consolidation"),
    },
    "policy": {
        "human_review_required": True,
        "not_clinical_diagnosis": True,
        "not_real_3d_reconstruction": True,
        "final_product_scope": "MVP técnico demostrable con inferencia 2D multiplanar, agente y revisión profesional",
    },
}

config_path = REPO_CONFIG_ROOT / "data_freeze_config.json"
pf_config_path = PF_ROOT / "PF001_canonical_data_paths.json"

for path in [config_path, pf_config_path]:
    path.write_text(json.dumps(canonical_config, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Wrote:", path)

## PF-001 - Inventario físico resumido

Se genera un inventario de directorios y extensiones. Esto permite detectar si faltan rutas, si cambian conteos grandes o si se rompió una carpeta de datos.

In [4]:
def safe_rel(path: Path, root: Path) -> str:
    try:
        return str(path.relative_to(root))
    except Exception:
        return str(path)

def scan_dataset_root(name: str, root: Path, max_files_for_detailed_rows: int = 200000):
    rows = []
    if not root.exists():
        return pd.DataFrame([{
            "dataset_key": name,
            "root": str(root),
            "exists": False,
            "relative_path": "",
            "suffix": "",
            "size_bytes": 0,
            "mtime": None,
        }])
    files = [p for p in root.rglob("*") if p.is_file()]
    for p in files[:max_files_for_detailed_rows]:
        try:
            st = p.stat()
            rows.append({
                "dataset_key": name,
                "root": str(root),
                "exists": True,
                "relative_path": safe_rel(p, root),
                "suffix": p.suffix.lower(),
                "size_bytes": int(st.st_size),
                "mtime": datetime.datetime.fromtimestamp(st.st_mtime).isoformat(timespec="seconds"),
            })
        except Exception as e:
            rows.append({
                "dataset_key": name,
                "root": str(root),
                "exists": True,
                "relative_path": safe_rel(p, root),
                "suffix": p.suffix.lower(),
                "size_bytes": None,
                "mtime": None,
                "error": str(e),
            })
    if len(files) > max_files_for_detailed_rows:
        rows.append({
            "dataset_key": name,
            "root": str(root),
            "exists": True,
            "relative_path": f"TRUNCATED_AFTER_{max_files_for_detailed_rows}_FILES",
            "suffix": "",
            "size_bytes": 0,
            "mtime": None,
        })
    return pd.DataFrame(rows)

inventory_parts = [
    scan_dataset_root("spider_sagittal", SPIDER_ROOT),
    scan_dataset_root("alkafri_sudirman_axial", AXIAL_ROOT),
    scan_dataset_root("ssmspine_optional", SSMSPINE_ROOT),
]
physical_inventory_df = pd.concat(inventory_parts, ignore_index=True)
physical_inventory_path = PF_ROOT / "PF001_physical_file_inventory.csv"
physical_inventory_df.to_csv(physical_inventory_path, index=False)

summary_df = (
    physical_inventory_df
    .groupby(["dataset_key", "exists", "suffix"], dropna=False)
    .agg(file_count=("relative_path", "count"), total_size_bytes=("size_bytes", "sum"))
    .reset_index()
    .sort_values(["dataset_key", "suffix"])
)
summary_path = PF_ROOT / "PF001_physical_file_inventory_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("Inventory:", physical_inventory_path)
print("Summary:", summary_path)
display(summary_df)

Inventory: /content/drive/MyDrive/PFI_MVP/results/PF001_PF003_dataset_freeze/PF001_physical_file_inventory.csv
Summary: /content/drive/MyDrive/PFI_MVP/results/PF001_PF003_dataset_freeze/PF001_physical_file_inventory_summary.csv


,dataset_key,exists,suffix,file_count,total_size_bytes
0,alkafri_sudirman_axial,True,,1,35147
1,alkafri_sudirman_axial,True,.bat,1,173
2,alkafri_sudirman_axial,True,.csv,5,25059
3,alkafri_sudirman_axial,True,.docx,1,16923
4,alkafri_sudirman_axial,True,.ima,17497,4891858350
5,alkafri_sudirman_axial,True,.m,55,210777
6,alkafri_sudirman_axial,True,.mat,32,1657664069
7,alkafri_sudirman_axial,True,.md,1,3211
8,alkafri_sudirman_axial,True,.npy,1436,421047808
9,alkafri_sudirman_axial,True,.png,29361,1057796603


## PF-002 - Congelamiento de splits

La prioridad es no depender de splits implícitos. El notebook intenta:

1. detectar artefactos de splits ya existentes;
2. si no hay un split formal, generar un split estable por identificador de caso usando hash determinístico;
3. guardar un manifiesto final para el producto.

Esto no obliga a reentrenar inmediatamente. Deja una referencia para entrenamientos finales y reportes posteriores.

In [5]:
def stable_split(key: str, train_ratio=0.70, val_ratio=0.15) -> str:
    h = hashlib.md5(str(key).encode("utf-8")).hexdigest()
    v = int(h[:8], 16) / 0xFFFFFFFF
    if v < train_ratio:
        return "train"
    if v < train_ratio + val_ratio:
        return "val"
    return "test"

def extract_case_id_from_path(p: Path) -> str:
    stem = p.stem
    # Common examples: 101_t1, 1_t2, patient_001, L1_0001_DY
    return re.sub(r"(_t1|_t2|_T1|_T2|\.nii|\.mha)$", "", stem)

split_source_rows = []
split_manifest_rows = []

# 1) Search likely existing split artifacts.
candidate_split_files = []
for base in [RESULTS_ROOT, DATA_ROOT, REPO_ROOT]:
    if base.exists():
        for pat in ["*split*.csv", "*splits*.csv", "*train*.csv", "*val*.csv", "*test*.csv"]:
            candidate_split_files.extend(base.rglob(pat))

candidate_split_files = sorted(set(candidate_split_files))
for p in candidate_split_files[:200]:
    split_source_rows.append({
        "path": str(p),
        "relative_to_pfi": safe_rel(p, PFI_ROOT),
        "size_bytes": p.stat().st_size if p.exists() else None,
        "source_type": "discovered_candidate_split_artifact",
    })

split_sources_df = pd.DataFrame(split_source_rows)
split_sources_path = PF_ROOT / "PF002_discovered_split_sources.csv"
split_sources_df.to_csv(split_sources_path, index=False)
print("Discovered split-like files:", len(split_sources_df), "->", split_sources_path)
display(split_sources_df.head(20))

Discovered split-like files: 12 -> /content/drive/MyDrive/PFI_MVP/results/PF001_PF003_dataset_freeze/PF002_discovered_split_sources.csv


,path,relative_to_pfi,size_bytes,source_type
0,/content/drive/MyDrive/PFI_MVP/results/E10_axi...,results/E10_axial_t2_final_training_clean/E10_...,1262,discovered_candidate_split_artifact
1,/content/drive/MyDrive/PFI_MVP/results/E11_axi...,results/E11_axial_class_mapping_final_report/E...,1116,discovered_candidate_split_artifact
2,/content/drive/MyDrive/PFI_MVP/results/E5_base...,results/E5_baseline_mejorado_binario/E5_improv...,95659,discovered_candidate_split_artifact
3,/content/drive/MyDrive/PFI_MVP/results/E5_base...,results/E5_baseline_mejorado_binario/E5_improv...,2412,discovered_candidate_split_artifact
4,/content/drive/MyDrive/PFI_MVP/results/E5_base...,results/E5_baseline_sagital/E5_training_histor...,649,discovered_candidate_split_artifact
5,/content/drive/MyDrive/PFI_MVP/results/E5_hold...,results/E5_holdout_validacion/E5_holdout_vs_in...,611,discovered_candidate_split_artifact
6,/content/drive/MyDrive/PFI_MVP/results/E5_mult...,results/E5_multiclase_agrupado/E5_multiclass_s...,15025,discovered_candidate_split_artifact
7,/content/drive/MyDrive/PFI_MVP/results/E5_mult...,results/E5_multiclase_agrupado/E5_multiclass_t...,3246,discovered_candidate_split_artifact
8,/content/drive/MyDrive/PFI_MVP/results/E5_mult...,results/E5_multiclase_holdout/E5_multiclass_ho...,352,discovered_candidate_split_artifact
9,/content/drive/MyDrive/PFI_MVP/results/E7_ssms...,results/E7_ssmspine_inventory/E7_ssmspine_sche...,7775,discovered_candidate_split_artifact


In [6]:
# 2) Generate/freeze a deterministic manifest for SPIDER sagittal.
spider_files = []
if SPIDER_ROOT.exists():
    for suffix in ["*.mha", "*.nii", "*.nii.gz"]:
        spider_files.extend(SPIDER_ROOT.rglob(suffix))

# Avoid t2_SPACE if present, keep likely T1/T2 series and labels separated when possible.
for p in sorted(set(spider_files)):
    rel = safe_rel(p, SPIDER_ROOT)
    lower = rel.lower()
    if "t2_space" in lower:
        continue
    if any(token in lower for token in ["mask", "label", "seg"]):
        role = "label_or_mask"
    else:
        role = "image_or_volume"
    case_id = extract_case_id_from_path(p)
    # Use case_id not file path, to keep image/mask in same split when names align.
    split = stable_split(case_id)
    split_manifest_rows.append({
        "dataset_key": "spider_sagittal",
        "case_id": case_id,
        "split": split,
        "modality_or_sequence": "sagittal_T1_T2_or_unknown",
        "file_role": role,
        "relative_path": rel,
        "source": "deterministic_hash_freeze_pf002",
        "notes": "Excluir t2_SPACE si aparece; revisar correspondencia imagen-máscara antes de reentrenar final."
    })

# 3) Generate/freeze a deterministic manifest for axial Al-Kafri/Sudirman.
# Prefer E8 final candidate CSV if available.
e8_candidates = []
if RESULTS_ROOT.exists():
    e8_candidates.extend(RESULTS_ROOT.rglob("*E8*FINAL*LABELS*FIXED*3*candidates*.csv"))
    e8_candidates.extend(RESULTS_ROOT.rglob("*final*candidates*.csv"))
e8_candidates = sorted(set(e8_candidates))

loaded_axial_from_csv = False
for csv_path in e8_candidates:
    try:
        df = pd.read_csv(csv_path)
        # Heuristic: must have enough rows and some path/case columns.
        if len(df) > 100:
            cols = {c.lower(): c for c in df.columns}
            case_col = None
            for cand in ["patient_id", "patient", "case_id", "case", "subject_id", "id"]:
                if cand in cols:
                    case_col = cols[cand]
                    break
            # Use index if no case col.
            seq_col = None
            for cand in ["sequence", "seq", "series", "contrast"]:
                if cand in cols:
                    seq_col = cols[cand]
                    break
            path_cols = [c for c in df.columns if "path" in c.lower() or "file" in c.lower()]
            for i, row in df.iterrows():
                seq = str(row[seq_col]) if seq_col else "unknown"
                # Keep all, but mark T2 if available.
                if seq_col and "t1" in seq.lower():
                    continue
                case_id = str(row[case_col]) if case_col else f"axial_row_{i:06d}"
                split = stable_split(case_id)
                rel_path = ""
                if path_cols:
                    rel_path = str(row[path_cols[0]])
                split_manifest_rows.append({
                    "dataset_key": "alkafri_sudirman_axial",
                    "case_id": case_id,
                    "split": split,
                    "modality_or_sequence": seq,
                    "file_role": "official_candidate_pair",
                    "relative_path": rel_path,
                    "source": f"deterministic_hash_freeze_from_{safe_rel(csv_path, PFI_ROOT)}",
                    "notes": "Filtrado T2 si la columna de secuencia lo permite; revisar mapeo final antes de reentrenar."
                })
            loaded_axial_from_csv = True
            print("Loaded axial candidates from:", csv_path)
            break
    except Exception as e:
        print("Could not load candidate CSV:", csv_path, e)

if not loaded_axial_from_csv and AXIAL_ROOT.exists():
    axial_files = []
    for suffix in ["*.png", "*.dcm", "*.ima", "*.csv"]:
        axial_files.extend(AXIAL_ROOT.rglob(suffix))
    for p in sorted(set(axial_files)):
        rel = safe_rel(p, AXIAL_ROOT)
        lower = rel.lower()
        role = "label_or_mask" if any(t in lower for t in ["label", "mask", "ground", "truth"]) else "image_or_metadata"
        case_id = extract_case_id_from_path(p)
        split_manifest_rows.append({
            "dataset_key": "alkafri_sudirman_axial",
            "case_id": case_id,
            "split": stable_split(case_id),
            "modality_or_sequence": "axial_T1_T2_or_unknown",
            "file_role": role,
            "relative_path": rel,
            "source": "deterministic_hash_freeze_pf002_from_files",
            "notes": "Fallback por archivos porque no se detectó CSV oficial de candidates."
        })

split_manifest_df = pd.DataFrame(split_manifest_rows)
split_manifest_path = PF_ROOT / "PF002_final_splits_manifest.csv"
split_manifest_df.to_csv(split_manifest_path, index=False)

if len(split_manifest_df):
    split_summary_df = (
        split_manifest_df
        .groupby(["dataset_key", "split", "file_role"], dropna=False)
        .agg(rows=("case_id", "count"), unique_cases=("case_id", "nunique"))
        .reset_index()
        .sort_values(["dataset_key", "split", "file_role"])
    )
else:
    split_summary_df = pd.DataFrame(columns=["dataset_key", "split", "file_role", "rows", "unique_cases"])

split_summary_path = PF_ROOT / "PF002_split_summary.csv"
split_summary_df.to_csv(split_summary_path, index=False)

print("Final split manifest:", split_manifest_path)
print("Split summary:", split_summary_path)
display(split_summary_df)

Loaded axial candidates from: /content/drive/MyDrive/PFI_MVP/results/E8_alkafri_official_pairing/E8_FINAL_LABELS_FIXED_3_candidates.csv
Final split manifest: /content/drive/MyDrive/PFI_MVP/results/PF001_PF003_dataset_freeze/PF002_final_splits_manifest.csv
Split summary: /content/drive/MyDrive/PFI_MVP/results/PF001_PF003_dataset_freeze/PF002_split_summary.csv


,dataset_key,split,file_role,rows,unique_cases
0,alkafri_sudirman_axial,test,official_candidate_pair,159,24
1,alkafri_sudirman_axial,train,official_candidate_pair,856,132
2,alkafri_sudirman_axial,val,official_candidate_pair,186,28
3,spider_sagittal,test,image_or_volume,58,30
4,spider_sagittal,test,label_or_mask,58,30
5,spider_sagittal,train,image_or_volume,279,150
6,spider_sagittal,train,label_or_mask,279,150
7,spider_sagittal,val,image_or_volume,69,36
8,spider_sagittal,val,label_or_mask,69,36


## PF-003 - Inventario de datasets y licencias

Este inventario funciona como documento de control. En la entrega final conviene verificar las licencias contra la página oficial de cada dataset y dejar la cita exacta en bibliografía.

In [7]:
dataset_inventory = [
    {
        "dataset_key": "spider_sagittal",
        "name": "SPIDER Lumbar Spine MRI Dataset",
        "root": str(SPIDER_ROOT),
        "exists": SPIDER_ROOT.exists(),
        "role_in_project": "Dataset principal sagital para segmentación de vértebras, discos y canal.",
        "plane": "sagittal",
        "format_observed": "MHA/NIfTI u otros según descarga local",
        "license_status": "verificar licencia/citación exacta en fuente oficial antes de entrega final",
        "privacy_status": "dataset público/anónimo según fuente; no usar datos privados identificables",
        "final_scope": "MVP principal sagital",
        "notes": "Base del modelo sagital consolidado E12.",
    },
    {
        "dataset_key": "alkafri_sudirman_axial",
        "name": "Al-Kafri / Sudirman Lumbar Spine MRI Dataset + Ground Truth",
        "root": str(AXIAL_ROOT),
        "exists": AXIAL_ROOT.exists(),
        "role_in_project": "Dataset complementario axial para segmentación T2 de regiones anotadas.",
        "plane": "axial",
        "format_observed": "DICOM/IMA, PNG labels, CSV metadata según descarga local",
        "license_status": "CC BY 4.0 según documentación del dataset en Mendeley; verificar cita final",
        "privacy_status": "dataset público/anónimo según fuente; no usar datos privados identificables",
        "final_scope": "MVP complementario axial",
        "notes": "Base del modelo axial T2 E10 y análisis de clases E11.",
    },
    {
        "dataset_key": "ssmspine_optional",
        "name": "SSMSpine u otro dataset opcional local",
        "root": str(SSMSPINE_ROOT),
        "exists": SSMSPINE_ROOT.exists(),
        "role_in_project": "Referencia opcional si está presente; no forma parte del núcleo final salvo decisión posterior.",
        "plane": "unknown_or_mixed",
        "format_observed": "a relevar",
        "license_status": "no definido en MVP; verificar antes de usar",
        "privacy_status": "no usar si no está clara la autorización",
        "final_scope": "fuera del núcleo actual",
        "notes": "Mantener separado de métricas finales si no se usa.",
    },
]

dataset_inventory_df = pd.DataFrame(dataset_inventory)
dataset_inventory_path = PF_ROOT / "PF003_dataset_inventory.csv"
dataset_inventory_df.to_csv(dataset_inventory_path, index=False)
display(dataset_inventory_df)
print("Dataset inventory:", dataset_inventory_path)

,dataset_key,name,root,exists,role_in_project,plane,format_observed,license_status,privacy_status,final_scope,notes
0,spider_sagittal,SPIDER Lumbar Spine MRI Dataset,/content/drive/MyDrive/PFI_MVP/data/SPIDER,True,Dataset principal sagital para segmentación de...,sagittal,MHA/NIfTI u otros según descarga local,verificar licencia/citación exacta en fuente o...,dataset público/anónimo según fuente; no usar ...,MVP principal sagital,Base del modelo sagital consolidado E12.
1,alkafri_sudirman_axial,Al-Kafri / Sudirman Lumbar Spine MRI Dataset +...,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI,True,Dataset complementario axial para segmentación...,axial,"DICOM/IMA, PNG labels, CSV metadata según desc...",CC BY 4.0 según documentación del dataset en M...,dataset público/anónimo según fuente; no usar ...,MVP complementario axial,Base del modelo axial T2 E10 y análisis de cla...
2,ssmspine_optional,SSMSpine u otro dataset opcional local,/content/drive/MyDrive/PFI_MVP/data/SSMSPINE,True,Referencia opcional si está presente; no forma...,unknown_or_mixed,a relevar,no definido en MVP; verificar antes de usar,no usar si no está clara la autorización,fuera del núcleo actual,Mantener separado de métricas finales si no se...


Dataset inventory: /content/drive/MyDrive/PFI_MVP/results/PF001_PF003_dataset_freeze/PF003_dataset_inventory.csv


In [8]:
license_md = f'''# PF-003 - Inventario de datasets y licencias

Fecha de congelamiento: {freeze_timestamp}

## Criterio general

El producto final se apoya en datasets públicos y documentados. No se incorporan datos clínicos privados identificables en esta etapa. Antes de la entrega final, cada dataset usado en métricas o demo debe tener ruta local, rol, fuente, licencia y cita bibliográfica verificadas.

## Datasets del MVP técnico

### 1. SPIDER

- Rol: dataset principal sagital.
- Uso: entrenamiento/evaluación del modelo sagital multiclase.
- Estado local: {SPIDER_ROOT.exists()}.
- Ruta local: `{SPIDER_ROOT}`.
- Licencia/cita: verificar contra la fuente oficial antes de la entrega final.
- Decisión: mantener como base del módulo sagital.

### 2. Al-Kafri / Sudirman

- Rol: dataset complementario axial.
- Uso: entrenamiento/evaluación del modelo axial T2.
- Estado local: {AXIAL_ROOT.exists()}.
- Ruta local: `{AXIAL_ROOT}`.
- Licencia: CC BY 4.0 según documentación del dataset en Mendeley; verificar cita final.
- Decisión: mantener como base del módulo axial complementario.

### 3. SSMSpine / opcionales

- Rol: referencia opcional si existe localmente.
- Estado local: {SSMSPINE_ROOT.exists()}.
- Decisión: no incluir en métricas finales salvo decisión explícita y licencia verificada.

## Restricciones

- No mezclar métricas finales con datasets no documentados.
- No usar datos identificables privados.
- No presentar el sistema como diagnóstico autónomo.
- Mantener separados los resultados sagitales y axiales si provienen de datasets distintos.
- No afirmar reconstrucción 3D real a partir de datasets no emparejados por paciente.

## Evidencia generada

- `PF001_canonical_data_paths.json`
- `PF001_physical_file_inventory.csv`
- `PF001_physical_file_inventory_summary.csv`
- `PF002_final_splits_manifest.csv`
- `PF002_split_summary.csv`
- `PF003_dataset_inventory.csv`
- `PF003_dataset_licenses.md`
'''

license_path = PF_ROOT / "PF003_dataset_licenses.md"
license_repo_path = REPO_DOCS_ROOT / "PF001_PF003_dataset_licenses.md"
for p in [license_path, license_repo_path]:
    p.write_text(license_md, encoding="utf-8")
    print("Wrote:", p)

Wrote: /content/drive/MyDrive/PFI_MVP/results/PF001_PF003_dataset_freeze/PF003_dataset_licenses.md
Wrote: /content/drive/MyDrive/PFI_MVP/repo/docs/PF001_PF003_dataset_licenses.md


## Reporte final PF-001 a PF-003

In [9]:
checks = []

def add_check(name, ok, detail):
    checks.append({"check": name, "ok": bool(ok), "detail": detail})

add_check("spider_root_exists", SPIDER_ROOT.exists(), str(SPIDER_ROOT))
add_check("axial_root_exists", AXIAL_ROOT.exists(), str(AXIAL_ROOT))
add_check("canonical_config_written", config_path.exists(), str(config_path))
add_check("physical_inventory_written", physical_inventory_path.exists(), str(physical_inventory_path))
add_check("split_manifest_written", split_manifest_path.exists(), str(split_manifest_path))
add_check("dataset_inventory_written", dataset_inventory_path.exists(), str(dataset_inventory_path))
add_check("license_doc_written", license_path.exists(), str(license_path))
add_check("has_split_rows", len(split_manifest_df) > 0, f"rows={len(split_manifest_df)}")
add_check("has_dataset_inventory_rows", len(dataset_inventory_df) >= 2, f"rows={len(dataset_inventory_df)}")

checks_df = pd.DataFrame(checks)
checks_path = PF_ROOT / "PF001_PF003_checks.csv"
checks_df.to_csv(checks_path, index=False)

all_ok = bool(checks_df["ok"].all())

report = {
    "notebook": "28_PF001_PF003_dataset_freeze_inventory",
    "goal": "standardize final data paths, freeze splits and document dataset inventory/licenses",
    "timestamp": freeze_timestamp,
    "outputs": {
        "canonical_config_repo": str(config_path),
        "canonical_config_results": str(pf_config_path),
        "physical_inventory_csv": str(physical_inventory_path),
        "physical_inventory_summary_csv": str(summary_path),
        "discovered_split_sources_csv": str(split_sources_path),
        "final_splits_manifest_csv": str(split_manifest_path),
        "split_summary_csv": str(split_summary_path),
        "dataset_inventory_csv": str(dataset_inventory_path),
        "dataset_licenses_md": str(license_path),
        "checks_csv": str(checks_path),
    },
    "summary": {
        "datasets_documented": int(len(dataset_inventory_df)),
        "split_manifest_rows": int(len(split_manifest_df)),
        "split_manifest_datasets": sorted(split_manifest_df["dataset_key"].dropna().unique().tolist()) if len(split_manifest_df) else [],
        "all_checks_ok": all_ok,
    },
    "methodological_policy": canonical_config["policy"],
    "decision": "PF001_PF003_dataset_base_ready_for_final_training" if all_ok else "PF001_PF003_requires_review",
}

report_path = PF_ROOT / "PF001_PF003_report.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

# Backlog closure draft inside repo.
closure_md = f'''# PF-001 a PF-003 - Cierre de base de datos y splits

Estado: {"OK" if all_ok else "Requiere revisión"}.

## Objetivo

Estandarizar rutas finales de datasets, congelar un manifiesto de splits y documentar inventario/licencias para el producto final.

## Salidas principales

- `{config_path.relative_to(REPO_ROOT) if str(config_path).startswith(str(REPO_ROOT)) else config_path}`
- `{physical_inventory_path}`
- `{summary_path}`
- `{split_manifest_path}`
- `{split_summary_path}`
- `{dataset_inventory_path}`
- `{license_path}`
- `{report_path}`

## Resumen

- Datasets documentados: {len(dataset_inventory_df)}
- Filas en split manifest: {len(split_manifest_df)}
- Checks OK: {all_ok}

## Decisión

{report["decision"]}

## Próximo paso

PF-004 a PF-007: consolidación/reentrenamiento final de modelos y reporte definitivo de métricas.
'''

closure_path = REPO_BACKLOG_ROOT / "PF001_PF003_resultados_cierre.md"
closure_path.write_text(closure_md, encoding="utf-8")

print("Checks:", checks_path)
display(checks_df)
print("Report:", report_path)
print(json.dumps(report, indent=2, ensure_ascii=False))
print("Backlog closure draft:", closure_path)

Checks: /content/drive/MyDrive/PFI_MVP/results/PF001_PF003_dataset_freeze/PF001_PF003_checks.csv


,check,ok,detail
0,spider_root_exists,True,/content/drive/MyDrive/PFI_MVP/data/SPIDER
1,axial_root_exists,True,/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI
2,canonical_config_written,True,/content/drive/MyDrive/PFI_MVP/repo/config/dat...
3,physical_inventory_written,True,/content/drive/MyDrive/PFI_MVP/results/PF001_P...
4,split_manifest_written,True,/content/drive/MyDrive/PFI_MVP/results/PF001_P...
5,dataset_inventory_written,True,/content/drive/MyDrive/PFI_MVP/results/PF001_P...
6,license_doc_written,True,/content/drive/MyDrive/PFI_MVP/results/PF001_P...
7,has_split_rows,True,rows=2013
8,has_dataset_inventory_rows,True,rows=3


Report: /content/drive/MyDrive/PFI_MVP/results/PF001_PF003_dataset_freeze/PF001_PF003_report.json
{
  "notebook": "28_PF001_PF003_dataset_freeze_inventory",
  "goal": "standardize final data paths, freeze splits and document dataset inventory/licenses",
  "timestamp": "2026-06-30T21:43:36",
  "outputs": {
    "canonical_config_repo": "/content/drive/MyDrive/PFI_MVP/repo/config/data_freeze_config.json",
    "canonical_config_results": "/content/drive/MyDrive/PFI_MVP/results/PF001_PF003_dataset_freeze/PF001_canonical_data_paths.json",
    "physical_inventory_csv": "/content/drive/MyDrive/PFI_MVP/results/PF001_PF003_dataset_freeze/PF001_physical_file_inventory.csv",
    "physical_inventory_summary_csv": "/content/drive/MyDrive/PFI_MVP/results/PF001_PF003_dataset_freeze/PF001_physical_file_inventory_summary.csv",
    "discovered_split_sources_csv": "/content/drive/MyDrive/PFI_MVP/results/PF001_PF003_dataset_freeze/PF002_discovered_split_sources.csv",
    "final_splits_manifest_csv": "/cont

## Commit sugerido

Ejecutar en una celda final o terminal de Colab:

```bash
%cd /content/drive/MyDrive/PFI_MVP/repo
!git status
!git add config/data_freeze_config.json docs/PF001_PF003_dataset_licenses.md backlogProducto/PF001_PF003_resultados_cierre.md notebooks/28_PF001_PF003_dataset_freeze_inventory.ipynb
!git commit -m "Add PF001 PF003 dataset freeze and inventory"
!git push
```

Los CSV/JSON de resultados quedan en Drive bajo `results/PF001_PF003_dataset_freeze/`. Si se quieren versionar también en GitHub, copiar primero esa carpeta a `repo/results/` o `repo/docs/evidencias/`.